[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C08/E08-1_Defauts_Roulement.ipynb)

# Défauts de roulement : décider par une suite de questions

## Le contexte

Un roulement à billes tourne à une vitesse qui varie d'un relevé à l'autre. Chaque relevé de maintenance conditionnelle donne la vitesse de rotation, la vitesse vibratoire efficace, le kurtosis et le facteur de crête du signal, ainsi que l'amplitude du spectre à trois fréquences caractéristiques : celle d'un défaut de bague extérieure (BPFO), de bague intérieure (BPFI) et de bille (BSF). Le roulement est sain, ou porte un défaut de bague extérieure, de bague intérieure ou de bille. Le service maintenance voudrait des règles lisibles, de la forme « si ceci, alors cela », pour décider quel élément changer.

La grandeur à prédire est un état parmi quatre : c'est un problème de classification multiclasse, et l'arbre de décision en est l'outil naturel.

Défi : prédire `etat` et obtenir un F-score macro supérieur à celui du modèle de départ sur la base de test. Ce modèle de départ est un arbre de décision aux réglages par défaut ; il atteint un F-score macro de 0,809 : à vous de faire mieux. La règle de découpage est fixée plus bas. La base de test sert à mesurer, jamais à apprendre ni à régler.

## Les données

Fichier `MecaIA-Defauts_Roulement.xlsx`, dans le dossier `donnees` du dépôt : 500 relevés, 8 colonnes. Une ligne est un relevé.

| Colonne | Signification | Unité |
|---|---|---|
| `vitesse_rpm` | vitesse de rotation de l'arbre | tr/min |
| `rms_mm_s` | vitesse vibratoire efficace globale | mm/s |
| `kurtosis` | aplatissement du signal temporel (3 pour un bruit gaussien) | sans dimension |
| `facteur_crete` | rapport de la crête à la valeur efficace | sans dimension |
| `amplitude_BPFO_mm_s` | amplitude spectrale à la fréquence de défaut de bague extérieure | mm/s |
| `amplitude_BPFI_mm_s` | amplitude spectrale à la fréquence de défaut de bague intérieure | mm/s |
| `amplitude_BSF_mm_s` | amplitude spectrale à la fréquence de défaut de bille | mm/s |
| `etat` | cible : `sain`, `bague_ext`, `bague_int` ou `bille` | catégorie |

Origine : jeu simulé, d'après les fréquences cinématiques d'un roulement de neuf billes. Le plancher de bruit de chaque raie croît avec la vitesse, et un défaut le relève. Quelques valeurs efficaces et kurtosis ne sont pas relevés. Il enseigne une méthode, et aucune conclusion de maintenance n'en découle.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions, ses cases vides et la répartition des quatre états avant toute modélisation : la forme d'un fichier se lit sur le fichier entier. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Defauts_Roulement.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Defauts_Roulement.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

cible = "etat"
print("dimensions :", df.shape)
vides = df.isna().sum()
print("cases vides :", vides[vides > 0].to_dict())
print(df[cible].value_counts().to_dict())
df.head(3)

Lecture du résultat : 500 relevés, 8 colonnes. La vitesse efficace compte 10 cases vides et le kurtosis 13, qu'il faudra remplir. Les roulements sains sont les plus nombreux (203 relevés), les trois défauts étant à peu près à égalité.

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas le dossier `donnees/` du dépôt. La cellule suivante ne télécharge rien : elle cherche ce dossier (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-IA_Ed2026`), vérifie la présence du classeur du défi et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez le dossier `donnees` à la racine de l'espace de travail et téléversez-y le classeur listé, pris dans le dépôt.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
from pathlib import Path

JEUX_LUS = ["MecaIA-Defauts_Roulement.xlsx"]                  # le classeur lu par ce notebook


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-IA_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees = chercher("donnees")
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Étape 2 : mettre le test de côté

Un quart des relevés est mis de côté avant tout apprentissage, tout réglage et toute comparaison avec la cible. La graine 42 fixe le tirage : chacun obtient la même base de test, et les scores se comparent. Le tirage est stratifié, pour que chaque état soit représenté dans les deux bases dans la même proportion.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=cible)
y = df[cible]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=GRAINE, stratify=y)
print("apprentissage :", len(X_train), "relevés")
print("test :", len(X_test), "relevés")

### Étape 3 : construire le modèle de départ

Le modèle de départ est celui de la section du livre : un arbre de décision, qui pose des questions successives sur une seule colonne à la fois, « cette amplitude dépasse-t-elle tel seuil ? », jusqu'à des feuilles qui portent chacune un état. Il garde ses réglages par défaut, donc aucune limite de profondeur. Les cases vides sont remplies par la médiane de l'apprentissage. Il apprend sur `X_train` seul.

In [ ]:
# Votre code :

### Étape 4 : mesurer sur la base de test

Le modèle prédit les relevés qu'il n'a jamais vus. On lit le F-score macro, l'exactitude et le F-score de chaque état, puis la taille de l'arbre et son score sur l'apprentissage. Un témoin sans apprentissage, qui répond toujours l'état le plus fréquent, donne le point de comparaison le plus bas.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

y_pred = modele.predict(X_test)
score_depart = f1_score(y_test, y_pred, average="macro")
temoin = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
score_temoin = f1_score(y_test, temoin.predict(X_test), average="macro")
print(f"F-score macro : {score_depart:.3f}")
print(f"exactitude : {accuracy_score(y_test, y_pred):.3f}")
print(f"F-score macro du témoin (toujours l'état le plus fréquent) : {score_temoin:.3f}")
classes = list(modele.classes_)
print(pd.Series(f1_score(y_test, y_pred, average=None, labels=classes), index=classes).round(3).to_string())
arbre_depart = modele[-1]
score_apprentissage = f1_score(y_train, modele.predict(X_train), average="macro")
print(f"arbre : profondeur {arbre_depart.get_depth()}, {arbre_depart.get_n_leaves()} feuilles, F-score macro sur l'apprentissage {score_apprentissage:.3f}")

Lecture du résultat : le F-score macro est de 0,809 pour une exactitude de 0,824, bien au-dessus du témoin (0,145). L'arbre compte 61 feuilles pour 375 relevés d'apprentissage, qu'il classe tous correctement (F-score macro de 1,000) : il a appris l'apprentissage presque par cœur, et l'écart avec le test est la trace de ce surapprentissage. C'est ce chiffre, mesuré sur des relevés que le modèle n'a jamais vus, que l'on a le droit d'annoncer.

## Pour aller plus loin : tenir compte de la vitesse

Le modèle de départ de l'étape 3 reste le score de référence du défi. Les trois cellules suivantes montrent comment le dépasser sans regarder le test pour choisir. La première lit, sur l'apprentissage, comment l'amplitude de chaque raie dépend de la vitesse pour les roulements sains : un seuil fixe sur une amplitude ne peut pas convenir à toutes les vitesses. La deuxième compare par validation croisée l'arbre de départ, un arbre élagué et ces deux arbres appliqués aux amplitudes rapportées à la tendance des roulements sains, apprise dans chaque pli. Une forêt aléatoire, vue à la section suivante, est donnée pour comparaison. La troisième mesure le modèle retenu une seule fois sur le test. Les valeurs sont lues dans les sorties imprimées.

In [ ]:
raies = ["amplitude_BPFO_mm_s", "amplitude_BPFI_mm_s", "amplitude_BSF_mm_s"]

# Votre code :

In [ ]:
# Votre code :

In [ ]:
# Votre code :

## Le score de référence

Le score publié est le F-score macro du modèle de départ de l'étape 3 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

score = f1_score(y_test, modele.predict(X_test), average="macro")
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE f1_macro = %.3f" % score)

## Synthèse

1. Un arbre de décision pose une question sur une seule colonne à la fois ; ses règles se lisent, mais ses frontières sont des marches alignées sur les axes.
2. Sans limite de profondeur, il classe l'apprentissage sans erreur : l'écart avec le test mesure le surapprentissage. Élaguer, c'est exiger un nombre minimal de relevés par feuille ; ici, sur les amplitudes brutes, cela ne suffit pas : 0,732 contre 0,781 en validation croisée.
3. Un arbre ne voit que ce qu'on lui montre : quand une grandeur dépend d'une autre, ici l'amplitude des raies de la vitesse, un seuil unique ne convient pas à toutes les vitesses, et c'est à l'ingénieur de construire la variable qui le rend valable.
4. Une grandeur construite à partir de l'état des roulements sains s'apprend dans chaque pli de la validation croisée, jamais sur tout l'apprentissage d'un coup, sans quoi les relevés de validation auraient servi à construire la variable.
5. Le choix du modèle se fait par validation croisée sur l'apprentissage ; le test ne sert qu'une fois, au modèle retenu.